In [21]:
import pandas as pd
from datetime import datetime, timedelta
import preprocessamento as pp
import numpy as np

In [5]:
papel = 'ENGI11.SA'
base = datetime.today().date() #- timedelta(days=1)
df_base = pp.montar_dataset(papel, dias_total_historico=600, data_base=base)

[ENGI11.SA] Cache existente cobre de 2020-01-02 a 2025-07-15.
[ENGI11.SA] Cache cobre período solicitado (2023-01-27 a 2025-07-15).
[ENGI11.SA] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[ENGI11.SA] Últimos dias atualizados em memória com sucesso.
[ENGI11.SA] Retornando 613 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20220127_20250716.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2025-07-15.
[^BVSP] Cache cobre período solicitado (2023-01-27 a 2025-07-15).
[^BVSP] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Últimos dias atualizados em memória com sucesso.
[^BVSP] Retornando 613 linhas para o período solicitado.
[BOVA11.SA] Cache existent

In [12]:
df_feriados = pd.read_csv('feriados_nacionais.csv', encoding='ISO-8859-1', sep=';')
df_feriados.head()


,Data,Dia da Semana,Feriado,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,01/01/2001,segunda-feira,Confraternização Universal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26/02/2001,segunda-feira,Carnaval,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27/02/2001,terça-feira,Carnaval,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13/04/2001,sexta-feira,Paixão de Cristo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21/04/2001,sábado,Tiradentes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_feriados['dt'] = pd.to_datetime(df_feriados['Data'], format='%d/%m/%Y')

In [14]:
df_feriados = df_feriados[['dt']]
df_feriados

,dt
0,2001-01-01
1,2001-02-26
2,2001-02-27
3,2001-04-13
4,2001-04-21
...,...
1259,2099-10-12
1260,2099-11-02
1261,2099-11-15
1262,2099-11-20


In [15]:
df_feriados.to_csv('feriados.csv', index=False)

In [31]:
df_feriados = pd.read_csv('feriados.csv', parse_dates=['dt'])
df_feriados.head()

,dt
0,2001-01-01
1,2001-02-26
2,2001-02-27
3,2001-04-13
4,2001-04-21


In [22]:
def features_temporais(df):
    """
    Adiciona features temporais ao DataFrame.
    """
    # transforma o índice em datetime
    df.index = pd.to_datetime(df.index)
    df['dia_semana'] = df.index.dayofweek
    df['feriado'] = df.index.isin(df_feriados['dt']).astype(int)
    # verifica se o dia seguinte é fim de semana ou feriado
    df['proximo_dia_util'] = np.where((df['dia_semana']==4) | (df.index + pd.Timedelta(days=1)).isin(df_feriados['dt']),False,True)

   
    return df

In [32]:
df_1 = features_temporais(df_base)
df_1.tail(10)

,open,high,low,close,adj_close,volume,pct_change,vol,rsi,macd,...,dolar,open_ibov,high_ibov,low_ibov,close_ibov,adj_close_ibov,volume_ibov,dia_semana,feriado,proximo_dia_util
data,,,,,,,,,,,,,,,,,,,,,
2025-07-02,48.500000,48.500000,46.279999,46.950001,46.950001,3630600,-0.025731,0.019577,50.512198,0.236196,...,5.4512,139586.00000,140049.000000,138384.0000,139051.000000,139051.000000,3619807,2,0,True
2025-07-03,47.200001,48.389999,46.770000,48.330002,48.330002,1773500,0.029393,0.021706,57.767822,0.314390,...,5.4208,139051.00000,141304.000000,139051.0000,140928.000000,140928.000000,3879378,3,0,True
2025-07-04,48.389999,48.639999,48.220001,48.279999,48.279999,1031300,-0.001035,0.021534,57.439207,0.368081,...,5.4090,140928.00000,141537.000000,140597.0000,141478.000000,141478.000000,5528911,4,0,False
2025-07-07,48.000000,48.450001,47.340000,47.599998,47.599998,1240500,-0.014085,0.021710,53.021940,0.351708,...,5.4552,141265.00000,141342.000000,139295.0000,139490.000000,139490.000000,3691359,0,0,True
2025-07-08,47.400002,47.580002,47.029999,47.349998,47.349998,1941200,-0.005252,0.021283,51.455224,0.314928,...,5.4571,139491.00000,139591.000000,138770.0000,139303.000000,139303.000000,2574588,1,0,True
2025-07-09,47.360001,47.419998,46.450001,46.639999,46.639999,1950900,-0.014995,0.020483,47.190493,0.225885,...,5.4626,139303.00000,139331.000000,137299.0000,137481.000000,137481.000000,8538960,2,0,True
2025-07-10,46.500000,46.500000,45.669998,46.119999,46.119999,2357800,-0.011149,0.020061,44.294843,0.112067,...,5.5433,137472.00000,137472.000000,136015.0000,136743.000000,136743.000000,5335029,3,0,True
2025-07-11,45.970001,46.139999,45.509998,45.759998,45.759998,1538300,-0.007806,0.020128,42.357076,-0.007102,...,5.5722,136742.00000,136742.000000,135528.0000,136187.000000,136187.000000,4964400,4,0,False
2025-07-14,45.660000,45.900002,45.320000,45.450001,45.450001,1520500,-0.006774,0.014356,40.705709,-0.125117,...,5.5595,136187.00000,136187.000000,134840.0000,135299.000000,135299.000000,4839858,0,0,True


In [30]:
df_base['2025-04-20':'2025-05-05'][['dia_semana', 'feriado', 'proximo_dia_util']]

,dia_semana,feriado,proximo_dia_util
data,,,
2025-04-22,1,0,True
2025-04-23,2,0,True
2025-04-24,3,0,True
2025-04-25,4,0,False
2025-04-28,0,0,True
2025-04-29,1,0,True
2025-04-30,2,0,False
2025-05-02,4,0,False
2025-05-05,0,0,True
